# QNN TEST

### IMPORTS

In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from scripts.snp       import SNP
from scripts.baskets   import config
from scripts.portfolio import Portfolio
from scripts.qaoa      import QAOA
from scripts.qnn       import QNN

### DATA 

In [3]:
tickers, start, end = config("mag7")
snp = SNP(tickers, start, end).cached_fetch(name="mag7")

✓ cached → data\mag7.parquet


### BRUTE FORCE

In [4]:
pf = Portfolio(snp.mu, snp.Sigma, lam=2.0, A=0.5, K=2, tickers=snp.tickers)
bf = pf.brute_force()
print(f"Brute-force: {bf['tickers']}  cost={bf['cost']:.6f}\n")

Brute-force: ['AAPL', 'MSFT']  cost=0.001247



### QAOA

In [5]:
# QAOA
qaoa = QAOA(pf, seed=42)
qaoa_res = qaoa.optimise(p=2, n_restarts=15)
print(f"QAOA p=2:   E={qaoa_res['energy']:.6f}, "
      f"ratio={qaoa.approximation_ratio(qaoa_res['energy']):.4f}")

QAOA p=2:   E=-0.500647, ratio=0.9991


### QNN

In [6]:
# QNN
qnn = QNN(n_layers=4, sharpness=5.0, seed=42)
qnn.fit(pf, n_epochs=150, lr=0.08, n_restarts=6)
print(f"\nQNN soft cost: {qnn.soft_cost():.6f}")
print(f"QNN hard cost: {qnn.hard_cost():.6f}")
for row in qnn.decode():
    mark = "  ←" if row["selected"] else ""
    print(f"  {row['ticker']:<6}  score={row['score']:+.3f}  "
          f"prob={row['prob']:.3f}{mark}")


QNN soft cost: 0.001190
QNN hard cost: 0.001247
  AAPL    score=+0.992  prob=0.797  ←
  MSFT    score=+0.965  prob=0.775  ←
  GOOGL   score=+0.659  prob=0.427
  AMZN    score=-0.994  prob=0.000
